# Paso 0: calibración del ambiente (sin entrenar)

**He(RL)l Kitchen v2.** Antes de entrenar DQN y PPO fijamos con datos los valores marcados como *(propuesta)* en el diseño (`docs/env_design.md`, sección 4.2). Solo se usan políticas escritas a mano, así que todo corre en minutos de CPU.

Todas las corridas usan las **seeds de validación** (`20000 + i`); las de prueba quedan reservadas para el reporte.

| Criterio | Pregunta |
|---|---|
| 1. Costo de probar | ¿Con qué costo la mejor frecuencia de prueba queda *en medio* (ni "nunca" ni "siempre")? |
| 2. Rango de σ_obs | ¿Entre qué ruidos la vista pasa de "casi alcanza" a "casi no sirve"? |
| 3. N por defecto | ¿Cuántas órdenes hay en la cola el 90 % del tiempo? |
| 4. Priorizar importa | ¿Una heurística que elige órdenes (EDF) supera claramente a la FIFO? |
| 5. Dificultad | ¿FIFO supera a la aleatoria sin acercarse al 100 % a tiempo? |

**Cómo correrlo**

- **Local:** con el entorno virtual activo, abrir este notebook desde `notebooks/` y ejecutar todo.
- **Google Colab:** abrir el notebook (desde GitHub o subiéndolo) y ejecutar todo; la primera celda clona el repo e instala las dependencias.

Las tablas y figuras se guardan en `results/` (o en la carpeta de la variable de entorno `KITCHEN_NB_OUTPUT_DIR`, que usan los tests).

In [ ]:
# Arranque: en Colab clona el repo; en local no hace nada extra.
import os, subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules and not Path("/content/PROY-RL").exists():
    subprocess.run(["git", "clone", "https://github.com/paulabaal12/PROY-RL.git",
                    "/content/PROY-RL"], check=True)
sys.path.insert(0, "/content/PROY-RL/notebooks" if "google.colab" in sys.modules else ".")

from _setup import setup
ROOT = setup()
print("Raíz del repo:", ROOT)

In [ ]:
%matplotlib inline
import os
from pathlib import Path

import pandas as pd

# Episodios por configuración. Los tests lo bajan con KITCHEN_NB_EPISODES.
EPISODES = int(os.environ.get("KITCHEN_NB_EPISODES", 50))
OUT = Path(os.environ.get("KITCHEN_NB_OUTPUT_DIR", "results"))
FIGURES, TABLES = OUT / "figures", OUT / "tables"
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)
print("Episodios por configuración:", EPISODES)

from experiments import calibrate

## 1 y 2. Costo de probar y ruido visual

La heurística FIFO prueba cada *k* pasos cuando el plato "se ve" casi listo ("nunca" = sirve solo con la lectura visual). Se barre *k* para varios costos de probar y varios σ_obs.

**Qué buscamos:** un costo con el que la mejor *k* sea **interior**. Si lo mejor es "nunca", probar no vale la pena; si es *k* = 1, conviene probar siempre y no hay decisión que aprender. Para σ_obs, buscamos un rango cuyos extremos se comporten distinto: con poco ruido casi no hace falta probar y con mucho la vista casi no sirve.

In [ ]:
sweep = calibrate.taste_sweep(n_episodes=EPISODES)
best = calibrate.best_retaste(sweep)
sweep.to_csv(TABLES / "calibration_taste_sweep.csv", index=False)
best.to_csv(TABLES / "calibration_best_retaste.csv", index=False)
best

In [ ]:
calibrate.plot_taste_sweep(sweep, out_dir=FIGURES);

## 3. Tamaño por defecto de la ventana visible N

Se corre la FIFO en la cocina randomizada y se mide cuántas órdenes esperan en la cola en cada paso. El N por defecto es el menor que cubre la cola en al menos el 90 % de los pasos. La ablación de N (1, 2, 4, 6) se hace alrededor de este valor.

In [ ]:
lengths = calibrate.queue_lengths(n_episodes=EPISODES)
lengths.to_csv(TABLES / "calibration_queue_lengths.csv", index=False)
print("N por defecto (cuantil 0.9):", calibrate.default_window(lengths))
calibrate.plot_queue_lengths(lengths, out_dir=FIGURES);

## 4. ¿Priorizar importa?

Comparamos la FIFO (solo atiende la orden más antigua) con una heurística EDF que elige la orden con la hora de entrega más próxima y adelanta preparación. La EDF **no** es un baseline del reporte: solo verifica que el ambiente premia elegir bien. Si no supera claramente a la FIFO, la pregunta central del proyecto (¿aprenden los agentes a priorizar?) no tendría respuesta y hay que ajustar los platos o las horas de entrega.

In [ ]:
prio = calibrate.prioritization_check(n_episodes=EPISODES)
prio.to_csv(TABLES / "calibration_prioritization.csv", index=False)
prio.groupby(["scenario", "policy"], sort=False)[["return", "on_time_rate", "expired_rate"]].agg(["mean", "std"]).round(3)

In [ ]:
calibrate.plot_prioritization(prio, out_dir=FIGURES);

## 5. Dificultad

La FIFO debe superar claramente a la aleatoria (el ambiente no se resuelve solo) y **no** estar cerca de 100 % a tiempo (debe quedar margen para que el RL mejore).

In [ ]:
diff = calibrate.difficulty_check(n_episodes=EPISODES)
diff.to_csv(TABLES / "calibration_difficulty.csv", index=False)
diff.groupby(["scenario", "policy"], sort=False)[["return", "on_time_rate", "expired_rate"]].mean().round(3)

In [ ]:
calibrate.plot_difficulty(diff, out_dir=FIGURES);

## Conclusiones

*Pendiente: se completa después de ejecutar el notebook con todos los episodios y de que el equipo decida los valores (Task 9 del plan). Los valores elegidos se registran en `docs/env_design.md`, sección 10.*